In [7]:
import os
print(os.getcwd())
os.chdir('/home/user1/MXY/EHRScore') # Change to the project directory

/home/user1/MXY/EHRScore


In [8]:
import pandas as pd

# read formatted mimic4 data
os.makedirs("data/mimic4/preprocessed", exist_ok=True)
data_path = "data/mimic4/format_mimic4_anonymized.csv"
mimic4_df = pd.read_csv(data_path)
mimic4_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10001,1,1,2111-06-04 22:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,2809;2731;4019;2113;53560;5533;56210;4240;V103,"Iron deficiency anemia, unspecified;Monoclonal...",Ferric Gluconate;Pneumococcal Vac Polyvalent;S...,4516;4542,Esophagogastroduodenoscopy [EGD] with closed b...
1,10001,1,2,2111-06-05 10:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10001,1,3,2111-06-05 22:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10001,1,4,2111-06-06 10:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,1,5,2111-06-06 22:33:00,2111-06-04 22:33:00,2111-06-07 20:59:00,0,2.93,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4082066,190677,3,15,2142-02-11 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,36.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4082067,190677,3,16,2142-02-12 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,36.83,98.0,NaN,NaN,7.34,NaN,NaN,NaN,NaN,NaN
4082068,190677,3,17,2142-02-12 18:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,35.40,96.0,NaN,NaN,7.23,NaN,NaN,NaN,NaN,NaN
4082069,190677,3,18,2142-02-13 06:26:00,2142-02-04 18:26:00,2142-02-13 19:36:00,1,9.05,0,1,...,35.70,97.0,NaN,75.0,7.23,NaN,NaN,NaN,NaN,NaN


In [9]:
import numpy as np

TASK                    = "outcome"        # "outcome" or "readmission"
RNG_SEED                = 42

DESIRED_TOTAL           = 20_000               # outcome: 20_000  readmission: 20_000

# Outcome
DESIRED_POS_RATIO_OUTCOME = 0.13              

# Readmission
TARGET_RATE_READM       = 0.17                
TOLERANCE_READM         = 0.005               
# ==========================================

if TASK == "outcome":
    filtered_df = (
    mimic4_df.groupby("PatientID")
             .filter(lambda x: len(x) >= 1) 
)
elif TASK == "readmission":
    filtered_df = (
    mimic4_df.groupby("PatientID")
             .filter(lambda x: len(x) >= 8) 
)

print(f"[INFO] {filtered_df['PatientID'].nunique():,} patients remaining after filtering")

rng = np.random.default_rng(RNG_SEED)

# Outcome TASK

if TASK.lower() == "outcome":
    label_col = "Outcome"

    patient_label_df = (
        filtered_df.groupby("PatientID")[label_col]
                   .max()
                   .reset_index()
    )
    pos_ids = patient_label_df.query(f"{label_col} == 1")["PatientID"].to_numpy()
    neg_ids = patient_label_df.query(f"{label_col} == 0")["PatientID"].to_numpy()

    desired_pos = int(DESIRED_TOTAL * DESIRED_POS_RATIO_OUTCOME)
    desired_neg = DESIRED_TOTAL - desired_pos

    sampled_pos = rng.choice(pos_ids, size=min(desired_pos, len(pos_ids)), replace=False)
    sampled_neg = rng.choice(neg_ids, size=min(desired_neg, len(neg_ids)), replace=False)

    if len(sampled_pos) < desired_pos:
        need = desired_pos - len(sampled_pos)
        extra_neg = rng.choice(
            [i for i in neg_ids if i not in sampled_neg], size=need, replace=False
        )
        sampled_neg = np.concatenate([sampled_neg, extra_neg])

    if len(sampled_neg) < desired_neg:
        need = desired_neg - len(sampled_neg)
        extra_pos = rng.choice(
            [i for i in pos_ids if i not in sampled_pos], size=need, replace=False
        )
        sampled_pos = np.concatenate([sampled_pos, extra_pos])

    selected_patients = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(selected_patients)

    print(
        f"[Outcome] Selected {len(selected_patients):,} patients "
        f"(Positive: {len(sampled_pos):,}, Negative: {len(sampled_neg):,})"
    )

    mimic4_filtered_df = (
        filtered_df[filtered_df["PatientID"].isin(selected_patients)]
        .groupby(["PatientID", "VisitID"])
        .head(4)
        .reset_index(drop=True)
    )



# Readmission TASK

elif TASK.lower() == "readmission":
    readmit_col = "Readmission"
    visit_labels   = {}  
    patient_labels = {}  
    processed_parts = []

    for pid, grp in filtered_df.groupby("PatientID"):
        grp = grp.sort_values("VisitID")        
        visits = grp["VisitID"].unique()

        visit_flag = (
            grp.groupby("VisitID")[readmit_col]
               .max()
               .reindex(visits, fill_value=0)
               .to_numpy()
        )

        last_one_idx = np.where(visit_flag == 1)[0][-1] if visit_flag.any() else None
        keep_visits  = visits if last_one_idx is None else visits[: last_one_idx + 1]

        has_positive = False
        for vid, flag in zip(visits, visit_flag):
            if vid in keep_visits:
                visit_labels[(pid, vid)] = int(flag)
                if flag == 1:
                    has_positive = True
        patient_labels[pid] = int(has_positive)

        processed_parts.append(grp[grp["VisitID"].isin(keep_visits)])

    processed_df = pd.concat(processed_parts, ignore_index=True)

    pos_pids = [pid for pid, lab in patient_labels.items() if lab == 1]
    neg_pids = [pid for pid, lab in patient_labels.items() if lab == 0]

    desired_pos = int(DESIRED_TOTAL * TARGET_RATE_READM)
    desired_neg = DESIRED_TOTAL - desired_pos

    sampled_pos = rng.choice(pos_pids, size=min(desired_pos, len(pos_pids)), replace=False)
    sampled_neg = rng.choice(neg_pids, size=min(desired_neg, len(neg_pids)), replace=False)

    if len(sampled_pos) < desired_pos and len(neg_pids) > len(sampled_neg):
        need = desired_pos - len(sampled_pos)
        extra_neg = rng.choice(
            [i for i in neg_pids if i not in sampled_neg],
            size=min(need, len(neg_pids) - len(sampled_neg)),
            replace=False,
        )
        sampled_neg = np.concatenate([sampled_neg, extra_neg])

    if len(sampled_neg) < desired_neg and len(pos_pids) > len(sampled_pos):
        need = desired_neg - len(sampled_neg)
        extra_pos = rng.choice(
            [i for i in pos_pids if i not in sampled_pos],
            size=min(need, len(pos_pids) - len(sampled_pos)),
            replace=False,
        )
        sampled_pos = np.concatenate([sampled_pos, extra_pos])

    selected_pids = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(selected_pids)

    actual_rate = len(sampled_pos) / len(selected_pids)
    print(
        f"[Readmission] Selected {len(selected_pids):,} patients, "
        f"Readmission rate: {actual_rate:.3%} (Target: {TARGET_RATE_READM:.3%})"
    )

    mimic4_filtered_df = (
        processed_df[processed_df["PatientID"].isin(selected_pids)]
        .groupby(["PatientID", "VisitID"])
        .head(200)
        .reset_index(drop=True)
    )



[INFO] 106,822 patients remaining after filtering
[Readmission] Selected 20,000 patients, Readmission rate: 17.000% (Target: 17.000%)


In [10]:
# Perform LOCF(Last Observation Carried Forward) imputation
exclude_cols = [
    'Conditions_ICD9',
    'Conditions_Long',
    'Medications',
    'Procedures_ICD9',
    'Procedures_Long'
]

cols_to_impute = [col for col in mimic4_filtered_df.columns if col not in exclude_cols]

# locf
locf_imputed_df = mimic4_filtered_df.copy()
locf_imputed_df[cols_to_impute] = locf_imputed_df.groupby(['PatientID', 'VisitID'])[cols_to_impute].ffill()

# nocb
nocb_imputed_df = locf_imputed_df.copy()
nocb_imputed_df[cols_to_impute] = nocb_imputed_df.groupby(['PatientID', 'VisitID'])[cols_to_impute].bfill()

nocb_imputed_df

In [11]:
# Save the time-series data for subsequent statistical analysis
timeseries_df = nocb_imputed_df[cols_to_impute]
# timeseries_df = mimic4_filtered_df[cols_to_impute]
timeseries_df.to_csv("data/mimic4/preprocessed/timeseries_mimic4.csv", index=False)
timeseries_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Diastolic blood pressure,Systolic blood pressure,Mean blood pressure,Heart Rate,Respiratory rate,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH
0,10002,1,1,2122-01-19 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10002,1,2,2122-01-19 19:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10002,1,3,2122-01-20 07:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10002,1,4,2122-01-20 19:15:00,2122-01-19 07:15:00,2122-01-24 12:10:00,0,5.20,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10051,1,1,2164-10-29 22:23:00,2164-10-29 22:23:00,2164-11-03 17:20:00,0,4.79,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158917,190660,1,2,2178-09-26 18:18:00,2178-09-26 06:18:00,2178-09-28 15:15:00,0,2.37,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158918,190660,1,3,2178-09-27 06:18:00,2178-09-26 06:18:00,2178-09-28 15:15:00,0,2.37,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158919,190660,1,4,2178-09-27 18:18:00,2178-09-26 06:18:00,2178-09-28 15:15:00,0,2.37,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158920,190660,2,1,2178-11-27 23:35:00,2178-11-27 23:35:00,2178-11-28 14:20:00,0,0.61,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Extract the patient's conditions, medications, and procedures to CMPs.csv
clinical_cols = [
    'PatientID', 'VisitID',
    'AdmissionTime', 'DischargeTime',
    'Conditions_ICD9', 'Conditions_Long',
    'Medications',
    'Procedures_ICD9', 'Procedures_Long'
]
cmps_df = nocb_imputed_df.groupby(['PatientID', 'VisitID']).first().reset_index()[clinical_cols]
# cmps_df = mimic4_filtered_df.groupby(['PatientID', 'VisitID']).first().reset_index()[clinical_cols]

cmps_df.to_csv("data/mimic4/preprocessed/CMPs_mimic4.csv", index=False)
cmps_df

,PatientID,VisitID,AdmissionTime,DischargeTime,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10002,1,2122-01-19 07:15:00,2122-01-24 12:10:00,71595;2851;4580;2724;V1272,"Osteoarthrosis, unspecified whether generalize...",Enoxaparin Sodium;0.9% Sodium Chloride;CefazoL...,8151,Total hip replacement
1,10051,1,2164-10-29 22:23:00,2164-11-03 17:20:00,None,None,PNEUMOcoccal 23-valent polysaccharide vaccine;...,None,None
2,10056,1,2188-12-30 08:48:00,2189-01-02 15:45:00,85220;45341;4280;3310;29410;42731;496;7812;401...,Subdural hemorrhage following injury without m...,Fluticasone-Salmeterol Diskus (250/50) ;Predni...,387,Interruption of the vena cava
3,10056,2,2191-08-22 06:11:00,2191-08-29 13:34:00,82021;51851;5070;42833;2761;2875;4168;49121;51...,Closed fracture of intertrochanteric section o...,Phytonadione;Levofloxacin;MethylPREDNISolone S...,7915;9671;9604,Closed reduction of fracture with internal fix...
4,10068,1,2148-10-27 18:14:00,2148-11-03 13:50:00,82022;2851;V420;7100;4019;73300;E8809;33818,Closed fracture of subtrochanteric section of ...,Bisacodyl;Bisacodyl;Docusate Sodium;Acetaminop...,7935;7935;9904,Open reduction of fracture with internal fixat...
...,...,...,...,...,...,...,...,...,...
44258,190610,2,2187-02-06 08:00:00,2187-02-09 12:20:00,None,None,Calcitriol;Fluticasone Propionate 110mcg;Albut...,None,None
44259,190616,1,2185-01-17 19:11:00,2185-01-22 14:25:00,03849;51881;5845;78552;4820;28749;2760;2762;55...,Other septicemia due to gram-negative organism...,Polyethylene Glycol;Iso-Osmotic Dextrose;Piper...,9672;966;3891,Continuous invasive mechanical ventilation for...
44260,190621,1,2113-06-08 01:46:00,2113-06-13 14:15:00,None,None,Pantoprazole (Granules for DR Suspension);Senn...,None,None
44261,190660,1,2178-09-26 06:18:00,2178-09-28 15:15:00,65811;6202;V270;64511;65961;66411;65441,"Premature rupture of membranes, delivered, wit...",Oxycodone-Acetaminophen (5mg-325mg);Bisacodyl;...,7569,Repair of other current obstetric laceration
